In [19]:
import numpy as np
from utils import relative_gini_value, relative_gini_variance, relative_gini_confint, gini_difference_wald_test

In [2]:
def estimate_true_relative_gini_mc(
    generate_model_and_target,
    *,
    n_big=1_000_000,
    seed=12345,
):
    """
    Получает очень точное Монте-Карло приближение истинного относительного Джини.

    Нужно только для проверки покрытия доверительного интервала.

    Parameters
    ----------
    generate_model_and_target : callable
        Функция вида:

            mu_hat, y = generate_model_and_target(n, rng)

        где mu_hat и y — массивы длины n.

    n_big : int
        Размер большой выборки для приближения истинного значения параметра.

    seed : int
        Seed для воспроизводимости.

    Returns
    -------
    float
        Приближённое истинное значение относительного Джини.
    """
    rng = np.random.default_rng(seed)

    mu_hat_big, y_big = generate_model_and_target(n_big, rng)

    theta_mc = relative_gini_value(
        y=y_big,
        mu_hat=mu_hat_big,
    )

    return theta_mc



def monte_carlo_check_relative_gini(
    generate_model_and_target,
    n_values,
    *,
    B=1000,
    alpha=0.05,
    theta_true=None,
    n_big_for_theta=1_000_000,
    seed=42,
    verbose=True,
):
    """
    Монте-Карло проверка оценки дисперсии и доверительного интервала
    для относительного коэффициента Джини.

    Проверка 1:
        Var_MC(sqrt(n) * G_hat) сравнивается с mean(sigma2_hat).

    Проверка 2:
        Проверяется покрытие доверительного интервала:

            G_hat +- z_{1-alpha/2} * sigma_hat / sqrt(n)

        Для этого нужно истинное значение параметра.
        Если theta_true не задан, оно приближается большой Монте-Карло выборкой.

    Parameters
    ----------
    generate_model_and_target : callable
        Функция вида:

            mu_hat, y = generate_model_and_target(n, rng)

        где mu_hat и y — массивы длины n.

    n_values : list[int]
        Размеры выборок, на которых проводится эксперимент.

    B : int
        Количество Монте-Карло повторов для каждого n.

    alpha : float
        Уровень значимости для доверительного интервала.

    theta_true : float or None
        Истинное значение относительного Джини.
        Если None, будет оценено через большую Монте-Карло выборку.

    n_big_for_theta : int
        Размер большой выборки для приближения theta_true, если оно не задано.

    seed : int
        Seed для воспроизводимости.

    verbose : bool
        Печатать ли прогресс.

    Returns
    -------
    list[dict]
        Список словарей с результатами для каждого n.
    """

    if not (0.0 < alpha < 1.0):
        raise ValueError("alpha must be in (0, 1).")

    rng = np.random.default_rng(seed)

    if theta_true is None:
        if verbose:
            print("theta_true не задан. Оцениваю его большой Монте-Карло выборкой...")

        theta_true = estimate_true_relative_gini_mc(
            generate_model_and_target,
            n_big=n_big_for_theta,
            seed=seed + 10_000,
        )

        if verbose:
            print(f"theta_true_mc ≈ {theta_true:.8f}")

    results = []

    for n in n_values:
        if verbose:
            print(f"\nЗапуск эксперимента для n={n}, B={B}")

        g_hats = np.empty(B)
        sigma2_hats = np.empty(B)
        ci_lowers = np.empty(B)
        ci_uppers = np.empty(B)

        failed = 0

        for b in range(B):
            try:
                mu_hat, y = generate_model_and_target(n, rng)

                g_hat = relative_gini_value(
                    y=y,
                    mu_hat=mu_hat,
                )

                sigma2_hat = relative_gini_variance(
                    y=y,
                    mu_hat=mu_hat,
                )

                ci_lower, ci_upper = relative_gini_confint(
                    y=y,
                    mu_hat=mu_hat,
                    alpha=alpha,
                )

                g_hats[b] = g_hat
                sigma2_hats[b] = sigma2_hat
                ci_lowers[b] = ci_lower
                ci_uppers[b] = ci_upper

            except Exception:
                failed += 1
                g_hats[b] = np.nan
                sigma2_hats[b] = np.nan
                ci_lowers[b] = np.nan
                ci_uppers[b] = np.nan

        valid_mask = (
            np.isfinite(g_hats)
            & np.isfinite(sigma2_hats)
            & np.isfinite(ci_lowers)
            & np.isfinite(ci_uppers)
        )

        valid_count = int(np.sum(valid_mask))

        if valid_count < 2:
            raise RuntimeError(
                f"Для n={n} осталось меньше двух успешных повторов. "
                f"failed={failed}, valid_count={valid_count}."
            )

        g_hats_valid = g_hats[valid_mask]
        sigma2_hats_valid = sigma2_hats[valid_mask]
        ci_lowers_valid = ci_lowers[valid_mask]
        ci_uppers_valid = ci_uppers[valid_mask]

        # ------------------------------------------------------------
        # Проверка 1:
        # Var_MC(G_hat) vs mean(sigma2_hat / n)
        # ------------------------------------------------------------
        #mc_var_g_hat = np.var(g_hats_valid, ddof=1)
        #mean_estimated_var_g_hat = np.mean(sigma2_hats_valid / n)

        #ratio_check_1 = mean_estimated_var_g_hat / mc_var_g_hat

        # ------------------------------------------------------------
        # Проверка 2:
        # Var_MC(sqrt(n) * G_hat) vs mean(sigma2_hat)
        # ------------------------------------------------------------
        mc_asymptotic_var = np.var(np.sqrt(n) * g_hats_valid, ddof=1)
        mean_sigma2_hat = np.mean(sigma2_hats_valid)

        ratio_check_2 = mean_sigma2_hat / mc_asymptotic_var

        # ------------------------------------------------------------
        # Проверка 4:
        # Coverage доверительного интервала
        # ------------------------------------------------------------
        covered = (
            (ci_lowers_valid <= theta_true)
            & (theta_true <= ci_uppers_valid)
        )

        coverage = np.mean(covered)

        # Дополнительные полезные величины
        mean_g_hat = np.mean(g_hats_valid)
        bias = mean_g_hat - theta_true

        mean_ci_length = np.mean(ci_uppers_valid - ci_lowers_valid)

        result = {
            "n": n,
            "B_requested": B,
            "B_valid": valid_count,
            "failed": failed,

            "theta_true": theta_true,
            "mean_g_hat": mean_g_hat,
            "bias": bias,

            # Проверка 1
            #"mc_var_g_hat": mc_var_g_hat,
            #"mean_estimated_var_g_hat": mean_estimated_var_g_hat,
            #"ratio_check_1": ratio_check_1,

            # Проверка 2
            "mc_asymptotic_var": mc_asymptotic_var,
            "mean_sigma2_hat": mean_sigma2_hat,
            "ratio_check_1": ratio_check_2,

            # Проверка 4
            "alpha": alpha,
            "nominal_coverage": 1.0 - alpha,
            "empirical_coverage": coverage,
            "coverage_error": coverage - (1.0 - alpha),
            "mean_ci_length": mean_ci_length,
        }

        results.append(result)

        if verbose:
            print(f"mean G_hat: {mean_g_hat:.8f}")
            print(f"bias: {bias:.8f}")
           #print("Проверка 1:")
           #print(f"  Var_MC(G_hat):              {mc_var_g_hat:.8g}")
           #print(f"  mean(sigma2_hat / n):       {mean_estimated_var_g_hat:.8g}")
           #print(f"  ratio:                      {ratio_check_1:.4f}")
            print("Проверка 1:")
            print(f"  Var_MC(sqrt(n) * G_hat):    {mc_asymptotic_var:.8g}")
            print(f"  mean(sigma2_hat):           {mean_sigma2_hat:.8g}")
            print(f"  ratio:                      {ratio_check_2:.4f}")
            print("Проверка 2:")
            print(f"  nominal coverage:           {1.0 - alpha:.4f}")
            print(f"  empirical coverage:         {coverage:.4f}")
            print(f"  mean CI length:             {mean_ci_length:.8g}")

    return results



def print_relative_gini_mc_results(results):
    """
    Печатает компактную таблицу результатов.
    """

    header = (
        f"{'n':>8} "
        f"{'B':>8} "
        f"{'mean_G':>12} "
        f"{'bias':>12} "
        f"{'ratio_2':>10} "
        f"{'coverage':>10} "
        f"{'CI_len':>12}"
    )

    print(header)
    print("-" * len(header))

    for r in results:
        print(
            f"{r['n']:8d} "
            f"{r['B_valid']:8d} "
            f"{r['mean_g_hat']:12.6f} "
            f"{r['bias']:12.6f} "
            
            f"{r['ratio_check_1']:10.4f} "
            f"{r['empirical_coverage']:10.4f} "
            f"{r['mean_ci_length']:12.6f}"
        )

### Проверка для симуляции Gamma GLM

In [3]:
import numpy as np


def generate_model_and_target_gamma(
    n,
    rng,
    *,
    beta0=2.0,
    beta1=0.5,
    x_mean=0.0,
    x_sd=1.0,
    gamma_shape=5.0,
):
    """
    Генерация данных из Gamma GLM с известными коэффициентами.

    Модель:
        X_i ~ N(x_mean, x_sd^2)

        mu_i = E[Y_i | X_i] = exp(beta0 + beta1 * X_i)

        Y_i | X_i ~ Gamma(shape=gamma_shape,
                          scale=mu_i / gamma_shape)

    Тогда:
        E[Y_i | X_i] = mu_i,
        Var[Y_i | X_i] = mu_i^2 / gamma_shape.

    Parameters
    ----------
    n : int
        Размер выборки.

    rng : np.random.Generator
        Генератор случайных чисел NumPy.

    beta0 : float
        Свободный коэффициент GLM.

    beta1 : float
        Коэффициент при X.

    x_mean : float
        Среднее распределения X.

    x_sd : float
        Стандартное отклонение X.

    gamma_shape : float
        Shape-параметр Gamma-распределения.
        Чем больше gamma_shape, тем меньше условная дисперсия.

    Returns
    -------
    mu_hat : np.ndarray, shape (n,)
        Истинные условные матожидания mu_i. Их используем как score / prediction.

    y : np.ndarray, shape (n,)
        Сгенерированные целевые значения.
    """

    if n <= 0:
        raise ValueError("n must be positive.")

    if x_sd <= 0:
        raise ValueError("x_sd must be positive.")

    if gamma_shape <= 0:
        raise ValueError("gamma_shape must be positive.")

    x = rng.normal(
        loc=x_mean,
        scale=x_sd,
        size=n,
    )

    eta = beta0 + beta1 * x
    mu = np.exp(eta)

    y = rng.gamma(
        shape=gamma_shape,
        scale=mu / gamma_shape,
        size=n,
    )

    mu_hat = mu

    return mu_hat, y






In [7]:
n_values = [200, 500, 1000, 2000, 5000, 10000, 20000, 50000]

results = monte_carlo_check_relative_gini(
    generate_model_and_target=generate_model_and_target_gamma,
    n_values=n_values,
    B=1000,
    alpha=0.05,
    theta_true=None,
    n_big_for_theta=1_000_000,
    seed=40,
    verbose=True,
)

print_relative_gini_mc_results(results)

theta_true не задан. Оцениваю его большой Монте-Карло выборкой...
theta_true_mc ≈ 0.75857384

Запуск эксперимента для n=200, B=1000
mean G_hat: 0.75757487
bias: -0.00099897
Проверка 1:
  Var_MC(sqrt(n) * G_hat):    0.24710891
  mean(sigma2_hat):           0.24075093
  ratio:                      0.9743
Проверка 2:
  nominal coverage:           0.9500
  empirical coverage:         0.9280
  mean CI length:             0.13470605

Запуск эксперимента для n=500, B=1000
mean G_hat: 0.75827592
bias: -0.00029792
Проверка 1:
  Var_MC(sqrt(n) * G_hat):    0.25372208
  mean(sigma2_hat):           0.24138893
  ratio:                      0.9514
Проверка 2:
  nominal coverage:           0.9500
  empirical coverage:         0.9420
  mean CI length:             0.085750612

Запуск эксперимента для n=1000, B=1000
mean G_hat: 0.75832464
bias: -0.00024920
Проверка 1:
  Var_MC(sqrt(n) * G_hat):    0.24788055
  mean(sigma2_hat):           0.2396554
  ratio:                      0.9668
Проверка 2:
  nomin

In [8]:
n_values = [200, 500, 1000, 2000, 5000, 10000, 20000, 50000]

results = monte_carlo_check_relative_gini(
    generate_model_and_target=lambda n, rng: generate_model_and_target_gamma(
        n, 
        rng,
        beta0=2.0,
        beta1=0.5,
        x_mean=0.0,
        x_sd=2.0,
        gamma_shape=5.0
    ),
    n_values=n_values,
    B=1000,
    alpha=0.05,
    theta_true=None,
    n_big_for_theta=1_000_000,
    seed=40,
    verbose=True,
)

print_relative_gini_mc_results(results)

theta_true не задан. Оцениваю его большой Монте-Карло выборкой...
theta_true_mc ≈ 0.92729561

Запуск эксперимента для n=200, B=1000
mean G_hat: 0.92631514
bias: -0.00098046
Проверка 1:
  Var_MC(sqrt(n) * G_hat):    0.044491427
  mean(sigma2_hat):           0.040424834
  ratio:                      0.9086
Проверка 2:
  nominal coverage:           0.9500
  empirical coverage:         0.9220
  mean CI length:             0.054598614

Запуск эксперимента для n=500, B=1000
mean G_hat: 0.92671613
bias: -0.00057947
Проверка 1:
  Var_MC(sqrt(n) * G_hat):    0.041688924
  mean(sigma2_hat):           0.041272491
  ratio:                      0.9900
Проверка 2:
  nominal coverage:           0.9500
  empirical coverage:         0.9470
  mean CI length:             0.035231792

Запуск эксперимента для n=1000, B=1000
mean G_hat: 0.92693824
bias: -0.00035736
Проверка 1:
  Var_MC(sqrt(n) * G_hat):    0.045668242
  mean(sigma2_hat):           0.041142054
  ratio:                      0.9009
Проверка 2:

In [9]:
n_values = [200, 500, 1000, 2000, 5000, 10000, 20000, 50000]

results = monte_carlo_check_relative_gini(
    generate_model_and_target=lambda n, rng: generate_model_and_target_gamma(
        n, 
        rng,
        beta0=2.0,
        beta1=0.5,
        x_mean=0.0,
        x_sd=0.5,
        gamma_shape=5.0
    ),
    n_values=n_values,
    B=1000,
    alpha=0.05,
    theta_true=None,
    n_big_for_theta=1_000_000,
    seed=40,
    verbose=True,
)

print_relative_gini_mc_results(results)

theta_true не задан. Оцениваю его большой Монте-Карло выборкой...
theta_true_mc ≈ 0.49791686

Запуск эксперимента для n=200, B=1000
mean G_hat: 0.49594204
bias: -0.00197482
Проверка 1:
  Var_MC(sqrt(n) * G_hat):    0.68242104
  mean(sigma2_hat):           0.65413628
  ratio:                      0.9586
Проверка 2:
  nominal coverage:           0.9500
  empirical coverage:         0.9350
  mean CI length:             0.22318228

Запуск эксперимента для n=500, B=1000
mean G_hat: 0.49871691
bias: 0.00080005
Проверка 1:
  Var_MC(sqrt(n) * G_hat):    0.70871403
  mean(sigma2_hat):           0.65638597
  ratio:                      0.9262
Проверка 2:
  nominal coverage:           0.9500
  empirical coverage:         0.9380
  mean CI length:             0.14172677

Запуск эксперимента для n=1000, B=1000
mean G_hat: 0.49784443
bias: -0.00007243
Проверка 1:
  Var_MC(sqrt(n) * G_hat):    0.64879594
  mean(sigma2_hat):           0.65679708
  ratio:                      1.0123
Проверка 2:
  nomina

### Проверка для симуляции Inverse Gaussian GLM

In [10]:
import numpy as np


def generate_model_and_target_inverse_gaussian(
    n,
    rng,
    *,
    beta0=2.0,
    beta1=0.5,
    x_mean=0.0,
    x_sd=1.0,
    phi=0.2,
):
    """
    Генерация данных из Inverse Gaussian GLM с известными коэффициентами.

    Модель:
        X_i ~ N(x_mean, x_sd^2)

        mu_i = E[Y_i | X_i] = exp(beta0 + beta1 * X_i)

        Y_i | X_i ~ InverseGaussian(mu_i, lambda_i)

    Параметризация NumPy:
        rng.wald(mean=mu, scale=lambda)

    Для Inverse Gaussian:
        E[Y_i | X_i] = mu_i
        Var[Y_i | X_i] = mu_i^3 / lambda_i

    В GLM обычно:
        Var[Y_i | X_i] = phi * mu_i^3

    Поэтому берем:
        lambda_i = 1 / phi

    Parameters
    ----------
    n : int
        Размер выборки.

    rng : np.random.Generator
        Генератор случайных чисел NumPy.

    beta0 : float
        Свободный коэффициент GLM.

    beta1 : float
        Коэффициент при X.

    x_mean : float
        Среднее распределения X.

    x_sd : float
        Стандартное отклонение X.

    phi : float
        Дисперсионный параметр GLM.
        Чем больше phi, тем больше условная дисперсия.

    Returns
    -------
    mu_hat : np.ndarray, shape (n,)
        Истинные условные матожидания mu_i.
        Их используем как score / prediction.

    y : np.ndarray, shape (n,)
        Сгенерированные целевые значения.
    """

    if n <= 0:
        raise ValueError("n must be positive.")

    if x_sd <= 0:
        raise ValueError("x_sd must be positive.")

    if phi <= 0:
        raise ValueError("phi must be positive.")

    x = rng.normal(
        loc=x_mean,
        scale=x_sd,
        size=n,
    )

    eta = beta0 + beta1 * x
    mu = np.exp(eta)

    # NumPy: Wald(mean, scale), где scale = lambda.
    # Нужно Var(Y|X) = phi * mu^3.
    # У Wald Var(Y|X) = mu^3 / lambda.
    # Значит lambda = 1 / phi.
    lam = 1.0 / phi

    y = rng.wald(
        mean=mu,
        scale=lam,
        size=n,
    )

    mu_hat = mu

    return mu_hat, y

In [11]:
results = monte_carlo_check_relative_gini(
    generate_model_and_target=generate_model_and_target_inverse_gaussian,
    n_values= [200, 500, 1000, 2000, 5000, 10000, 20000, 50000],
    B=1000,
    alpha=0.05,
    theta_true=None,
    n_big_for_theta=1_000_000,
    seed=42,
    verbose=True,
)

print_relative_gini_mc_results(results)

theta_true не задан. Оцениваю его большой Монте-Карло выборкой...
theta_true_mc ≈ 0.46957157

Запуск эксперимента для n=200, B=1000
mean G_hat: 0.45785594
bias: -0.01171563
Проверка 1:
  Var_MC(sqrt(n) * G_hat):    1.7586587
  mean(sigma2_hat):           1.4112438
  ratio:                      0.8025
Проверка 2:
  nominal coverage:           0.9500
  empirical coverage:         0.9070
  mean CI length:             0.32194463

Запуск эксперимента для n=500, B=1000
mean G_hat: 0.46239177
bias: -0.00717979
Проверка 1:
  Var_MC(sqrt(n) * G_hat):    1.8264916
  mean(sigma2_hat):           1.6683479
  ratio:                      0.9134
Проверка 2:
  nominal coverage:           0.9500
  empirical coverage:         0.9200
  mean CI length:             0.22161541

Запуск эксперимента для n=1000, B=1000
mean G_hat: 0.46368839
bias: -0.00588317
Проверка 1:
  Var_MC(sqrt(n) * G_hat):    1.7354308
  mean(sigma2_hat):           1.8147889
  ratio:                      1.0457
Проверка 2:
  nominal cov

In [12]:

results = monte_carlo_check_relative_gini(
    generate_model_and_target= lambda n, rng: generate_model_and_target_inverse_gaussian(
    n,
    rng,

    beta0=2.0,
    beta1=0.5,
    x_mean=0.0,
    x_sd=2.0,
    phi=0.2,
),
    n_values= [200, 500, 1000, 2000, 5000, 10000, 20000, 50000],
    B=1000,
    alpha=0.05,
    theta_true=None,
    n_big_for_theta=1_000_000,
    seed=42,
    verbose=True,
)

print_relative_gini_mc_results(results)

theta_true не задан. Оцениваю его большой Монте-Карло выборкой...
theta_true_mc ≈ 0.70888546

Запуск эксперимента для n=200, B=1000
mean G_hat: 0.67430836
bias: -0.03457710
Проверка 1:
  Var_MC(sqrt(n) * G_hat):    1.6614689
  mean(sigma2_hat):           0.92305861
  ratio:                      0.5556
Проверка 2:
  nominal coverage:           0.9500
  empirical coverage:         0.7970
  mean CI length:             0.25961804

Запуск эксперимента для n=500, B=1000
mean G_hat: 0.69091327
bias: -0.01797219
Проверка 1:
  Var_MC(sqrt(n) * G_hat):    2.206668
  mean(sigma2_hat):           1.3847077
  ratio:                      0.6275
Проверка 2:
  nominal coverage:           0.9500
  empirical coverage:         0.8220
  mean CI length:             0.19931198

Запуск эксперимента для n=1000, B=1000
mean G_hat: 0.69858862
bias: -0.01029684
Проверка 1:
  Var_MC(sqrt(n) * G_hat):    2.7258034
  mean(sigma2_hat):           1.8240401
  ratio:                      0.6692
Проверка 2:
  nominal cov

In [13]:

results = monte_carlo_check_relative_gini(
    generate_model_and_target= lambda n, rng: generate_model_and_target_inverse_gaussian(
    n,
    rng,

    beta0=2.0,
    beta1=0.5,
    x_mean=0.0,
    x_sd=0.5,
    phi=0.2,
),
    n_values= [200, 500, 1000, 2000, 5000, 10000, 20000, 50000],
    B=1000,
    alpha=0.05,
    theta_true=None,
    n_big_for_theta=1_000_000,
    seed=42,
    verbose=True,
)

print_relative_gini_mc_results(results)

theta_true не задан. Оцениваю его большой Монте-Карло выборкой...
theta_true_mc ≈ 0.26250753

Запуск эксперимента для n=200, B=1000
mean G_hat: 0.25742864
bias: -0.00507889
Проверка 1:
  Var_MC(sqrt(n) * G_hat):    1.8644357
  mean(sigma2_hat):           1.6300774
  ratio:                      0.8743
Проверка 2:
  nominal coverage:           0.9500
  empirical coverage:         0.9210
  mean CI length:             0.34870186

Запуск эксперимента для n=500, B=1000
mean G_hat: 0.25826307
bias: -0.00424446
Проверка 1:
  Var_MC(sqrt(n) * G_hat):    1.8512454
  mean(sigma2_hat):           1.7236555
  ratio:                      0.9311
Проверка 2:
  nominal coverage:           0.9500
  empirical coverage:         0.9310
  mean CI length:             0.22802121

Запуск эксперимента для n=1000, B=1000
mean G_hat: 0.25726054
bias: -0.00524699
Проверка 1:
  Var_MC(sqrt(n) * G_hat):    1.6991348
  mean(sigma2_hat):           1.7934577
  ratio:                      1.0555
Проверка 2:
  nominal cov

### Проверка для симуляции Пуассоновской GLM

In [14]:
def generate_model_and_target_poisson(
    n,
    rng,
    *,
    beta0=2.0,
    beta1=0.5,
    x_mean=0.0,
    x_sd=1.0,
):
    """
    Генерация данных из Poisson GLM с известными коэффициентами.

    Модель:
        X_i ~ N(x_mean, x_sd^2)

        lambda_i = E[Y_i | X_i] = exp(beta0 + beta1 * X_i)

        Y_i | X_i ~ Poisson(lambda_i)

    В Poisson GLM с log-link:
        log(lambda_i) = beta0 + beta1 * X_i

    Тогда:
        E[Y_i | X_i] = lambda_i,
        Var[Y_i | X_i] = lambda_i.

    Parameters
    ----------
    n : int
        Размер выборки.

    rng : np.random.Generator
        Генератор случайных чисел NumPy.

    beta0 : float
        Свободный коэффициент GLM.

    beta1 : float
        Коэффициент при X.

    x_mean : float
        Среднее распределения X.

    x_sd : float
        Стандартное отклонение X.

    Returns
    -------
    mu_hat : np.ndarray, shape (n,)
        Истинные условные матожидания lambda_i.
        Их используем как score / prediction.

    y : np.ndarray, shape (n,)
        Сгенерированные целевые значения.
    """

    if n <= 0:
        raise ValueError("n must be positive.")

    if x_sd <= 0:
        raise ValueError("x_sd must be positive.")

    x = rng.normal(
        loc=x_mean,
        scale=x_sd,
        size=n,
    )

    eta = beta0 + beta1 * x
    mu = np.exp(eta)

    y = rng.poisson(
        lam=mu,
        size=n,
    )

    mu_hat = mu

    return mu_hat, y

In [15]:
results = monte_carlo_check_relative_gini(
    generate_model_and_target=generate_model_and_target_poisson,
    n_values=[200, 500, 1000, 2000, 5000, 10000, 20000, 50000],
    B=1000,
    alpha=0.05,
    theta_true=None,
    n_big_for_theta=1_000_000,
    seed=42,
    verbose=True,
)

print_relative_gini_mc_results(results)

theta_true не задан. Оцениваю его большой Монте-Карло выборкой...
theta_true_mc ≈ 0.82400586

Запуск эксперимента для n=200, B=1000
mean G_hat: 0.82266603
bias: -0.00133983
Проверка 1:
  Var_MC(sqrt(n) * G_hat):    0.13169107
  mean(sigma2_hat):           0.13577612
  ratio:                      1.0310
Проверка 2:
  nominal coverage:           0.9500
  empirical coverage:         0.9470
  mean CI length:             0.10100205

Запуск эксперимента для n=500, B=1000
mean G_hat: 0.82310857
bias: -0.00089729
Проверка 1:
  Var_MC(sqrt(n) * G_hat):    0.14505322
  mean(sigma2_hat):           0.13374814
  ratio:                      0.9221
Проверка 2:
  nominal coverage:           0.9500
  empirical coverage:         0.9380
  mean CI length:             0.063800705

Запуск эксперимента для n=1000, B=1000
mean G_hat: 0.82378644
bias: -0.00021942
Проверка 1:
  Var_MC(sqrt(n) * G_hat):    0.15205667
  mean(sigma2_hat):           0.13302576
  ratio:                      0.8748
Проверка 2:
  nomi

In [17]:
results = monte_carlo_check_relative_gini(
    generate_model_and_target= lambda n, rng: generate_model_and_target_poisson(
    n,
    rng,
    beta0=2.0,
    beta1=0.5,
    x_mean=0.0,
    x_sd=2.0,
),
    n_values=[200, 500, 1000, 2000, 5000, 10000, 20000, 50000],
    B=1000,
    alpha=0.05,
    theta_true=None,
    n_big_for_theta=1_000_000,
    seed=42,
    verbose=True,
)

print_relative_gini_mc_results(results)

theta_true не задан. Оцениваю его большой Монте-Карло выборкой...
theta_true_mc ≈ 0.95979772

Запуск эксперимента для n=200, B=1000
mean G_hat: 0.95868229
bias: -0.00111543
Проверка 1:
  Var_MC(sqrt(n) * G_hat):    0.012200208
  mean(sigma2_hat):           0.013042416
  ratio:                      1.0690
Проверка 2:
  nominal coverage:           0.9500
  empirical coverage:         0.9500
  mean CI length:             0.031124244

Запуск эксперимента для n=500, B=1000
mean G_hat: 0.95966770
bias: -0.00013002
Проверка 1:
  Var_MC(sqrt(n) * G_hat):    0.012899407
  mean(sigma2_hat):           0.012650511
  ratio:                      0.9807
Проверка 2:
  nominal coverage:           0.9500
  empirical coverage:         0.9450
  mean CI length:             0.019553474

Запуск эксперимента для n=1000, B=1000
mean G_hat: 0.95947607
bias: -0.00032165
Проверка 1:
  Var_MC(sqrt(n) * G_hat):    0.011290139
  mean(sigma2_hat):           0.012917395
  ratio:                      1.1441
Проверка 2:

In [16]:
results = monte_carlo_check_relative_gini(
    generate_model_and_target= lambda n, rng: generate_model_and_target_poisson(
    n,
    rng,
    beta0=2.0,
    beta1=0.5,
    x_mean=0.0,
    x_sd=0.5,
),
    n_values=[200, 500, 1000, 2000, 5000, 10000, 20000, 50000],
    B=1000,
    alpha=0.05,
    theta_true=None,
    n_big_for_theta=1_000_000,
    seed=42,
    verbose=True,
)

print_relative_gini_mc_results(results)

theta_true не задан. Оцениваю его большой Монте-Карло выборкой...
theta_true_mc ≈ 0.57177785

Запуск эксперимента для n=200, B=1000
mean G_hat: 0.56667155
bias: -0.00510630
Проверка 1:
  Var_MC(sqrt(n) * G_hat):    0.55713683
  mean(sigma2_hat):           0.51846682
  ratio:                      0.9306
Проверка 2:
  nominal coverage:           0.9500
  empirical coverage:         0.9410
  mean CI length:             0.19848163

Запуск эксперимента для n=500, B=1000
mean G_hat: 0.57114432
bias: -0.00063353
Проверка 1:
  Var_MC(sqrt(n) * G_hat):    0.53285602
  mean(sigma2_hat):           0.51065832
  ratio:                      0.9583
Проверка 2:
  nominal coverage:           0.9500
  empirical coverage:         0.9390
  mean CI length:             0.12499154

Запуск эксперимента для n=1000, B=1000
mean G_hat: 0.57181549
bias: 0.00003764
Проверка 1:
  Var_MC(sqrt(n) * G_hat):    0.47914298
  mean(sigma2_hat):           0.50743615
  ratio:                      1.0590
Проверка 2:
  nomina

### Проверка стат теста

In [18]:
import numpy as np


def generate_two_gamma_glm_scores_one_target(
    n,
    rng,
    *,
    beta0=2.0,
    beta=None,
    n_features_a=3,
    n_features_b=1,
    x_mean=0.0,
    x_sd=1.0,
    gamma_shape=5.0,
    return_X=False,
):
    """
    Генерирует один таргет Y из Gamma GLM и две разные оценки матожидания.

    Истинная модель:
        X_i ~ N(x_mean, x_sd^2)

        mu_true_i = exp(beta0 + X_i @ beta)

        Y_i | X_i ~ Gamma(shape=gamma_shape,
                          scale=mu_true_i / gamma_shape)

    Тогда:
        E[Y_i | X_i] = mu_true_i
        Var[Y_i | X_i] = mu_true_i^2 / gamma_shape

    Две оценки:
        mu_hat_a использует первые n_features_a фичей
        mu_hat_b использует первые n_features_b фичей

    Обычно стоит брать:
        n_features_a > n_features_b,

    чтобы первая оценка была информативнее второй.

    Parameters
    ----------
    n : int
        Размер выборки.

    rng : np.random.Generator
        Генератор случайных чисел NumPy.

    beta0 : float
        Intercept истинной Gamma GLM.

    beta : array-like or None
        Коэффициенты при фичах. Если None, используется дефолтный вектор.

    n_features_a : int
        Сколько фичей использует первая оценка mu_hat_a.

    n_features_b : int
        Сколько фичей использует вторая оценка mu_hat_b.

    x_mean : float
        Среднее фичей.

    x_sd : float
        Стандартное отклонение фичей.

    gamma_shape : float
        Shape-параметр Gamma-распределения.
        Чем больше gamma_shape, тем меньше шум в Y.

    return_X : bool
        Если True, возвращает также матрицу X и истинное mu_true.

    Returns
    -------
    Если return_X=False:
        mu_hat_a, mu_hat_b, y

    Если return_X=True:
        mu_hat_a, mu_hat_b, y, X, mu_true
    """

    if n <= 0:
        raise ValueError("n must be positive.")

    if x_sd <= 0:
        raise ValueError("x_sd must be positive.")

    if gamma_shape <= 0:
        raise ValueError("gamma_shape must be positive.")

    if beta is None:
        beta = np.array([0.7, -0.5, 0.35, 0.25, -0.2], dtype=float)
    else:
        beta = np.asarray(beta, dtype=float)

    if beta.ndim != 1:
        raise ValueError("beta must be a one-dimensional array.")

    p_true = len(beta)

    if not (1 <= n_features_b <= p_true):
        raise ValueError("n_features_b must be between 1 and len(beta).")

    if not (1 <= n_features_a <= p_true):
        raise ValueError("n_features_a must be between 1 and len(beta).")

    if n_features_b > n_features_a:
        raise ValueError("Expected n_features_b <= n_features_a.")

    X = rng.normal(
        loc=x_mean,
        scale=x_sd,
        size=(n, p_true),
    )

    # Истинное матожидание, по которому генерируется Y
    eta_true = beta0 + X @ beta
    mu_true = np.exp(eta_true)

    y = rng.gamma(
        shape=gamma_shape,
        scale=mu_true / gamma_shape,
        size=n,
    )

    # Более информативная оценка: использует больше фичей
    eta_a = beta0 + X[:, :n_features_a] @ beta[:n_features_a]
    mu_hat_a = np.exp(eta_a)

    # Менее информативная оценка: использует меньше фичей
    eta_b = beta0 + X[:, :n_features_b] @ beta[:n_features_b]
    mu_hat_b = np.exp(eta_b)

    if return_X:
        return mu_hat_a, mu_hat_b, y, X, mu_true

    return mu_hat_a, mu_hat_b, y

In [20]:
rng = np.random.default_rng(42)

mu_hat_a, mu_hat_b, y = generate_two_gamma_glm_scores_one_target(
    n=1000,
    rng=rng,
    beta0=2.0,
    beta=np.array([0.7, -0.5, 0.35, 0.25, -0.2]),
    n_features_a=4,
    n_features_b=1,
    gamma_shape=5.0,
)


result = gini_difference_wald_test(
    y=y,
    score_a=mu_hat_a,
    score_b=mu_hat_b,
    score_c=y,
    alpha=0.05,
)

print("Gini A:", result["gini_A"])
print("Gini B:", result["gini_B"])
print("Gini C:", result["gini_C"])

print("Relative Gini A:", result["normalized_gini_A"])
print("Relative Gini B:", result["normalized_gini_B"])

print("Delta_hat:", result["delta_hat"])
print("T-stat:", result["t_stat"])
print("p-value:", result["p_value"])
print("Reject H0:", result["reject"])

Gini A: 0.4920431260500022
Gini B: 0.370341727211584
Gini C: 0.5416047830091694
Relative Gini A: 0.9084911017886486
Relative Gini B: 0.6837859244040577
Delta_hat: 0.22470517738459087
T-stat: 11.9171090218404
p-value: 0.0
Reject H0: True


In [21]:
import numpy as np


def generate_two_equal_quality_gamma_glm_scores(
    n,
    rng,
    *,
    beta0=2.0,
    block_beta=None,
    x_mean=0.0,
    x_sd=1.0,
    gamma_shape=5.0,
    return_X=False,
):
    """
    Генерирует один таргет Y из Gamma GLM и две score-функции одинакового качества.

    Конструкция:
        X_A и X_B независимы и одинаково распределены.

        eta_true = beta0 + X_A @ block_beta + X_B @ block_beta

        mu_true = exp(eta_true)

        Y | X ~ Gamma(shape=gamma_shape, scale=mu_true / gamma_shape)

    Две оценки:
        mu_hat_a = exp(beta0 + X_A @ block_beta)
        mu_hat_b = exp(beta0 + X_B @ block_beta)

    Так как X_A и X_B симметричны, mu_hat_a и mu_hat_b должны быть
    одинаковы по качеству предсказания в среднем.

    Parameters
    ----------
    n : int
        Размер выборки.

    rng : np.random.Generator
        Генератор случайных чисел NumPy.

    beta0 : float
        Intercept.

    block_beta : array-like or None
        Коэффициенты одного блока признаков.
        Один и тот же вектор используется для score A и score B.

    x_mean : float
        Среднее признаков.

    x_sd : float
        Стандартное отклонение признаков.

    gamma_shape : float
        Shape-параметр Gamma-распределения.

    return_X : bool
        Если True, возвращает также X_A, X_B и mu_true.

    Returns
    -------
    Если return_X=False:
        mu_hat_a, mu_hat_b, y

    Если return_X=True:
        mu_hat_a, mu_hat_b, y, X_A, X_B, mu_true
    """

    if n <= 0:
        raise ValueError("n must be positive.")

    if x_sd <= 0:
        raise ValueError("x_sd must be positive.")

    if gamma_shape <= 0:
        raise ValueError("gamma_shape must be positive.")

    if block_beta is None:
        block_beta = np.array([0.5, -0.4, 0.3], dtype=float)
    else:
        block_beta = np.asarray(block_beta, dtype=float)

    if block_beta.ndim != 1:
        raise ValueError("block_beta must be a one-dimensional array.")

    p_block = len(block_beta)

    X_A = rng.normal(
        loc=x_mean,
        scale=x_sd,
        size=(n, p_block),
    )

    X_B = rng.normal(
        loc=x_mean,
        scale=x_sd,
        size=(n, p_block),
    )

    eta_a = beta0 + X_A @ block_beta
    eta_b = beta0 + X_B @ block_beta

    # Истинная модель симметрично зависит от обоих блоков.
    # Чтобы средний уровень mu_true не был слишком большим,
    # intercept можно оставить beta0, а вклад каждого блока делить на sqrt(2).
    eta_true = beta0 + (X_A @ block_beta + X_B @ block_beta) / np.sqrt(2.0)

    mu_true = np.exp(eta_true)

    y = rng.gamma(
        shape=gamma_shape,
        scale=mu_true / gamma_shape,
        size=n,
    )

    mu_hat_a = np.exp(eta_a)
    mu_hat_b = np.exp(eta_b)

    if return_X:
        return mu_hat_a, mu_hat_b, y, X_A, X_B, mu_true

    return mu_hat_a, mu_hat_b, y

In [22]:
rng = np.random.default_rng(42)

mu_hat_a, mu_hat_b, y = generate_two_equal_quality_gamma_glm_scores(
    n=5000,
    rng=rng,
    beta0=2.0,
    block_beta=np.array([0.5, -0.4, 0.3]),
    gamma_shape=5.0,
)

result = gini_difference_wald_test(
    y=y,
    score_a=mu_hat_a,
    score_b=mu_hat_b,
    score_c=y,
    alpha=0.05,
)

print("Relative Gini A:", result["normalized_gini_A"])
print("Relative Gini B:", result["normalized_gini_B"])
print("Delta_hat:", result["delta_hat"])
print("T-stat:", result["t_stat"])
print("p-value:", result["p_value"])
print("Reject H0:", result["reject"])

Relative Gini A: 0.6229085034052184
Relative Gini B: 0.6197005543502329
Delta_hat: 0.003207949054985538
T-stat: 0.21152101995310849
p-value: 0.832480731701216
Reject H0: False
